In [1]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
CACHE_DIR = str(ROOT_DIR.parent / "huggingface_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import random
import json
from tqdm import tqdm
import torch
from dotenv import load_dotenv
from transformers import set_seed
from unsloth import FastLanguageModel, get_chat_template
load_dotenv()

SEED = 42
set_seed(SEED)
random.seed(SEED)

BATCH_SIZE = 8

sys.path.insert(0, str(ROOT_DIR))

DATA_DIR = (ROOT_DIR / "data/sft_data/train.json").resolve()
MODEL_DIR = (ROOT_DIR / "models").resolve()

/tmp/ipykernel_19343/3075473752.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import unsloth
from unsloth import FastLanguageModel, get_chat_template

def get_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-8B",
        max_seq_length=2048,
        load_in_4bit=True,
        device_map = "balanced"
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")
    return model, tokenizer

In [3]:
model, tokenizer = get_model_and_tokenizer()
model.load_adapter(MODEL_DIR / "continual-pretrain")

==((====))==  Unsloth 2026.4.2: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=str(DATA_DIR),
    split="train"
).shuffle(seed=SEED)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'output'],
    num_rows: 20000
})

In [5]:
def generate_conversation(examples):
    problems  = examples["instruction"]
    solutions = examples["output"]

    conversations = []

    for i, (problem, solution) in enumerate(zip(problems, solutions)):
        if problem is None or solution is None:
            print("\n=== LỖI DATA ===")
            print("Index:", i)
            print("Instruction:", problem)
            print("Output:", solution)
            print("================\n")

        conversations.append([
            {"role": "user", "content": problem if problem is not None else ""},
            {"role": "assistant", "content": solution if solution is not None else ""},
        ])

    return {"conversations": conversations}

In [6]:
dataset = dataset.map(generate_conversation, batched=True)

def format_chat(batch):
    texts = tokenizer.apply_chat_template(
        batch["conversations"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )
    texts = [t + tokenizer.eos_token for t in texts]

    return {"text": texts}

dataset = dataset.map(format_chat, batched=True)
dataset = dataset.remove_columns(["instruction", "output", "conversations"])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [10]:
dataset = dataset.train_test_split(test_size=0.05, seed=SEED)

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

/opt/conda/envs/dopn-venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/opt/conda/envs/dopn-venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.4.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [17]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

class CustomTrainer(UnslothTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(
                model, inputs, num_items_in_batch=num_items_in_batch
            )

        if not torch.isfinite(loss):
            print(f"[CustomTrainer] Skip step {self.state.global_step} (NaN/Inf)")

            self.accelerator.clear_gradients()

            return torch.zeros((), device=loss.device, dtype=loss.dtype)

        if self.args.n_gpu > 1:
            loss = loss.mean()

        self.accelerator.backward(loss)

        return loss.detach() / self.args.gradient_accumulation_steps


training_args = UnslothTrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    warmup_ratio=0.1,
    num_train_epochs=2,
    learning_rate=1e-5,
    max_grad_norm=1.0,

    eval_strategy="steps",
    eval_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,

    report_to="wandb",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,

    output_dir=MODEL_DIR / "sft1",
    gradient_checkpointing=False,
)

trainer = CustomTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=8,
    packing=False,
    args=training_args,
)

In [18]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [20]:
tokenizer.decode(trainer.train_dataset[10]["input_ids"])

'<|im_start|>user\nWhat action should the GANC take if the MS does not respond to the GA-PSR-ACTIVATE-UTC-REQ message during the network initiated GA-PSR TC activation procedure?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThe GANC aborts the network initiated activation procedure and responds to the MS with the GA-PSR-ACTIVATE-UTC-ACK message including the cause indicating successful activation.<|im_end|>\n<|im_end|>'

In [21]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                   <think>\n\n</think>\n\nHierarchical routing is not effective for Internet-like topologies because the Internet is a small-world network where most nodes are at small distances from each other. As a result, there are no remote nodes, making it impossible to efficiently apply hierarchical routing.<|im_end|>\n<|im_end|>'

In [24]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 19,000 | Num Epochs = 2 | Total steps = 2,376
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
questions = [

# ================= VI - MCQ =================
"""Trả lời nhanh: Biên độ của sóng điện từ thay đổi như thế nào khi nó truyền qua một vật dẫn?

A. Nó vẫn không đổi.
B. Nó tăng lên.
C. Nó giảm đi.
D. Nó dao động""",

# ================= VI - QNA =================
"""Hãy trả lời câu hỏi sau:

Mục đích của tính cước ngoại tuyến (offline charging) là gì trong 3GPP?

Trả lời ngắn gọn và giải thích.""",

# ================= EN - MCQ =================
"""Quick answer: How are voice calls delivered to dual/multi-mode terminals in hybrid networks? [3GPP Release 18]

1. Based on available radio access systems
2. Based on serving network conditions
3. Based on roaming agreements between operators
4. Based on CS core network domain
5. Based on service delivery options in the user profile""",

# ================= EN - QNA =================
"""Answer the following question:

What is the purpose of Tracking Area Update (TAU) in LTE networks?

Provide a concise explanation."""
]


messages_list = [
    [{"role": "user", "content": q}]
    for q in questions
]

In [ ]:
from transformers import TextStreamer

for messages in messages_list:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    print("\n===== =====\n")

    _ = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=512,
        temperature=0.5,
        top_p=0.8,
        top_k=20,
        streamer=TextStreamer(tokenizer, skip_prompt=False),
    )